### Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os

data_processed = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/processed")
print("Libraries loaded")

Libraries loaded


In [2]:
## Load all four expression datasets and assign disease labels to each sample

datasets = {
    "Vitiligo": pd.read_csv(f"{data_processed}/GSE65127_vitiligo_expr.csv", index_col=0),
    "SLE": pd.read_csv(f"{data_processed}/GSE318067_SLE_expr.csv", index_col=0),
    "RA": pd.read_csv(f"{data_processed}/GSE93272_RA_expr.csv", index_col=0),
    "T1D": pd.read_csv(f"{data_processed}/GSE55098_T1D_expr.csv", index_col=0)
}

# Build feature matrix — transpose so rows=samples, columns=genes
frames = []
for disease, df in datasets.items():
    transposed = df.T
    transposed['disease_label'] = disease
    frames.append(transposed)

feature_matrix = pd.concat(frames, axis=0)
print(f"Feature matrix shape: {feature_matrix.shape}")
print(feature_matrix['disease_label'].value_counts())

Feature matrix shape: (419, 21656)
disease_label
RA          275
SLE          82
Vitiligo     40
T1D          22
Name: count, dtype: int64


In [4]:
## Reduce features to DEG genes only — biologically meaningful feature selection

deg_genes_all = set()
for disease in ["Vitiligo", "SLE", "RA"]:
    sig = pd.read_csv(f"{data_processed}/{disease}_DEGs.csv", index_col=0)
    deg_genes_all.update(sig.index.tolist())

print(f"Total DEG genes for feature selection: {len(deg_genes_all)}")

# Keep only DEG columns plus label
available_genes = [g for g in deg_genes_all if g in feature_matrix.columns]
print(f"Available in feature matrix: {len(available_genes)}")

feature_matrix_filtered = feature_matrix[available_genes + ['disease_label']]
print(f"Filtered feature matrix shape: {feature_matrix_filtered.shape}")

Total DEG genes for feature selection: 11964
Available in feature matrix: 11964
Filtered feature matrix shape: (419, 11965)


In [5]:
## Further reduce using variance filtering
## Remove genes with near-zero variance — they carry no discriminative information

X = feature_matrix_filtered.drop(columns=['disease_label'])
y = feature_matrix_filtered['disease_label']

# Keep top 500 most variable genes
gene_variance = X.var()
top_genes = gene_variance.nlargest(500).index.tolist()

X_filtered = X[top_genes]
print(f"Final feature matrix: {X_filtered.shape}")

Final feature matrix: (419, 500)


In [6]:
## Scale features and encode labels for ML

# Standard scaling — brings all gene expression values to same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)
X_scaled = pd.DataFrame(X_scaled, columns=top_genes)

# Encode disease labels as numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Classes: {le.classes_}")
print(f"Encoded as: {list(range(len(le.classes_)))}")
print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y_encoded.shape}")

Classes: ['RA' 'SLE' 'T1D' 'Vitiligo']
Encoded as: [0, 1, 2, 3]
X shape: (419, 500)
y shape: (419,)


In [7]:
## Save final feature matrix for ML
X_scaled['disease_label'] = y_encoded
X_scaled.to_csv(f"{data_processed}/ML_feature_matrix.csv")

# Also save label mapping for reference
label_map = dict(zip(le.classes_, range(len(le.classes_))))
print(f"Label mapping: {label_map}")
print(f"Saved ML_feature_matrix.csv")

Label mapping: {'RA': 0, 'SLE': 1, 'T1D': 2, 'Vitiligo': 3}
Saved ML_feature_matrix.csv
